In [1]:
import requests
import os
import mysql.connector as mc
import pandas as pd
from datetime import datetime
import dotenv
import time
import calendar
 
dotenv.load_dotenv()
 
# -----------------------------------------------------------------------
# 기상청_지상(종관, ASOS) 시간자료 조회서비스
# https://www.data.go.kr/data/15057210/openapi.do
# dataCd=ASOS, dateCd=HR
# 한 번 호출 시 시작~종료 범위의 시간 데이터를 반환 (최대 numOfRows건)
# -----------------------------------------------------------------------
BASE_URL = 'https://apis.data.go.kr/1360000/AsosHourlyInfoService/getWthrDataList'
SERVICE_KEY = os.getenv('ASOS_API_KEY')
MAX_PAGE_SIZE = 999  
 
DB_CONFIG = {
    'host': os.getenv('DB_HOST'),
    'port': int(os.getenv('DB_PORT', 3306)),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'database': os.getenv('DB_DATABASE'),
    'autocommit': True,
}

In [2]:
INCHEON_STATIONS = {
    '112': '인천',
    '201': '강화',
    '232': '백령도',
}
 
# 날씨별 도로 패턴 분석에 필요한 관측 항목
# ta: 기온(℃), rn: 강수량(mm), hm: 습도(%), ws: 풍속(m/s), wd: 풍향(deg), pa: 현지기압(hPa)
WEATHER_COLS = ['ta', 'rn', 'hm', 'ws', 'wd', 'pa']
 
# DB 컬럼 구성: statDate + stnId + stnNm + 항목별 hour00~hour23
hour_cols = [f"hour{str(i).zfill(2)}" for i in range(24)]
columns = ["statDate", "stnId", "stnNm", "weatherItem"] + hour_cols
 
col_str = ", ".join(columns)
placeholders = ", ".join(["%s"] * len(columns))
insert_query = f"INSERT IGNORE INTO weather_pattern_asos ({col_str}) VALUES ({placeholders})"
 
 
def create_table():
    conn = mc.connect(**DB_CONFIG)
    cursor = conn.cursor()
    hour_col_defs = "\n".join([f"        {h} FLOAT," for h in hour_cols])
    # 마지막 컬럼 뒤 쉼표 제거
    hour_col_defs = hour_col_defs.rstrip(',')
    create_query = f"""
    CREATE TABLE IF NOT EXISTS weather_pattern_asos (
        id BIGINT AUTO_INCREMENT PRIMARY KEY,
        statDate VARCHAR(6) NOT NULL,
        stnId VARCHAR(10) NOT NULL,
        stnNm VARCHAR(50),
        weatherItem VARCHAR(10) NOT NULL,
{hour_col_defs},
        UNIQUE KEY uniq_asos (statDate, stnId, weatherItem)
    );
    """
    try:
        cursor.execute(create_query)
        conn.commit()
        print("[INFO] weather_pattern_asos 테이블 생성 완료")
    except Exception as e:
        print(f"[ERROR] 테이블 생성 실패: {e}")
    finally:
        cursor.close()
        conn.close()
 
 
create_table()

[INFO] weather_pattern_asos 테이블 생성 완료


In [3]:
def get_asos_hourly(start_dt, end_dt, stn_id):
    """
    특정 기간(start_dt~end_dt), 특정 지점(stn_id)의 ASOS 시간자료 호출.
    start_dt, end_dt: 'YYYYMMDD' 형식
    하루치(24건) 단위로 호출 권장.
    """
    params = {
        'ServiceKey': SERVICE_KEY,
        'pageNo': '1',
        'numOfRows': str(MAX_PAGE_SIZE),
        'dataType': 'JSON',
        'dataCd': 'ASOS',
        'dateCd': 'HR',
        'startDt': start_dt,
        'startHh': '01',
        'endDt': end_dt,
        'endHh': '23',
        'stnIds': stn_id,
    }
    try:
        response = requests.get(BASE_URL, params=params, timeout=15)
        return response.json()
    except Exception as e:
        print(f"\n[API 호출 에러] {start_dt}~{end_dt} stn={stn_id}: {e}")
        return None
 
 
def extract_items(json_data):
    """
    응답 JSON에서 item 리스트 추출.
    기존 요일별/시간대별 파싱 로직과 동일한 구조 유지.
    """
    try:
        if not json_data:
            return []
        body = json_data.get('response', {}).get('body', {})
        items_entry = body.get('items', [])
 
        if isinstance(items_entry, dict):
            items = items_entry.get('item', [])
        else:
            items = items_entry
 
        if not items:
            return []
        if isinstance(items, dict):
            items = [items]
 
        return [item for item in items if isinstance(item, dict)]
    except Exception as e:
        print(f"\n[파싱 에러 상세] {e}")
        return []
 
 
def parse_hourly_by_item(items):
    """
    item 리스트에서 관측 항목별 시간대 값을 딕셔너리로 변환.
    tm 필드 예시: '2023-01-01 01:00'  → hour = 1
    반환 형식: { 'ta': {1: -3.2, 2: -3.5, ...}, 'rn': {...}, ... }
    """
    result = {col: {} for col in WEATHER_COLS}
    for item in items:
        tm = item.get('tm', '')
        try:
            # tm: '2023-01-01 01:00' 형식
            hour = int(tm.strip().split(' ')[1].split(':')[0])
            # ASOS 시간자료는 01~24시 기준이므로 24시 → 0시로 변환
            if hour == 24:
                hour = 0
        except:
            continue
 
        for col in WEATHER_COLS:
            val = item.get(col, None)
            try:
                result[col][hour] = float(val) if val not in (None, '', ' ') else None
            except:
                result[col][hour] = None
 
    return result
 
 
def main():
    conn = mc.connect(**DB_CONFIG)
    cursor = conn.cursor()
 
    years = [2023, 2024, 2025, 2026]
    total_count = 0
 
    for year in years:
        for month in range(1, 13):
            if year == 2026 and month > 3:
                break
 
            target_ym = f"{year}{str(month).zfill(2)}"
            last_day = calendar.monthrange(year, month)[1]
 
            # 월별 전체 데이터 누적: { (stn_id, weather_item): { hour: [값 리스트] } }
            monthly_acc = {}
 
            print(f"\n[평균 작업 시작] {target_ym} ASOS 날씨 수집 중...")
 
            for day in range(1, last_day + 1):
                ymd = f"{year}{str(month).zfill(2)}{str(day).zfill(2)}"
 
                for stn_id, stn_nm in INCHEON_STATIONS.items():
                    data = get_asos_hourly(ymd, ymd, stn_id)
                    items = extract_items(data)
 
                    if not items:
                        time.sleep(0.2)
                        continue
 
                    hourly = parse_hourly_by_item(items)
 
                    for weather_item, hour_vals in hourly.items():
                        key = (stn_id, stn_nm, weather_item)
                        if key not in monthly_acc:
                            monthly_acc[key] = {h: [] for h in range(24)}
                        for h, v in hour_vals.items():
                            if v is not None:
                                monthly_acc[key][h].append(v)
 
                    time.sleep(0.2)  # API 호출 간격 (기존 코드와 동일)
 
                print(f"  - {ymd} 데이터 확보 완료", end="\r")
 
            # 월별 평균 계산 후 DB 적재
            if monthly_acc:
                final_rows = []
                for (stn_id, stn_nm, weather_item), hour_data in monthly_acc.items():
                    row = [target_ym, stn_id, stn_nm, weather_item]
                    for h in range(24):
                        vals = hour_data.get(h, [])
                        avg = round(sum(vals) / len(vals), 2) if vals else None
                        row.append(avg)
                    final_rows.append(tuple(row))
 
                if final_rows:
                    cursor.executemany(insert_query, final_rows)
                    total_count += len(final_rows)
                    print(f"  => {target_ym} 평균 데이터 {len(final_rows)}건 저장 성공")
 
    print(f"\n\n[최종 결과] 총 {total_count}건의 월별 ASOS 시간대 평균 데이터가 저장되었습니다.")
    cursor.close()
    conn.close()
 
 
if __name__ == "__main__":
    main()


[평균 작업 시작] 202301 ASOS 날씨 수집 중...
  => 202301 평균 데이터 18건 저장 성공

[평균 작업 시작] 202302 ASOS 날씨 수집 중...
  => 202302 평균 데이터 18건 저장 성공

[평균 작업 시작] 202303 ASOS 날씨 수집 중...
  - 20230325 데이터 확보 완료
[API 호출 에러] 20230326~20230326 stn=232: Expecting value: line 1 column 1 (char 0)
  => 202303 평균 데이터 18건 저장 성공

[평균 작업 시작] 202304 ASOS 날씨 수집 중...
  - 20230419 데이터 확보 완료
[API 호출 에러] 20230420~20230420 stn=201: Expecting value: line 1 column 1 (char 0)
  => 202304 평균 데이터 18건 저장 성공

[평균 작업 시작] 202305 ASOS 날씨 수집 중...
  => 202305 평균 데이터 18건 저장 성공

[평균 작업 시작] 202306 ASOS 날씨 수집 중...
  => 202306 평균 데이터 18건 저장 성공

[평균 작업 시작] 202307 ASOS 날씨 수집 중...
  => 202307 평균 데이터 18건 저장 성공

[평균 작업 시작] 202308 ASOS 날씨 수집 중...
  => 202308 평균 데이터 18건 저장 성공

[평균 작업 시작] 202309 ASOS 날씨 수집 중...
  => 202309 평균 데이터 18건 저장 성공

[평균 작업 시작] 202310 ASOS 날씨 수집 중...
  => 202310 평균 데이터 18건 저장 성공

[평균 작업 시작] 202311 ASOS 날씨 수집 중...
  => 202311 평균 데이터 18건 저장 성공

[평균 작업 시작] 202312 ASOS 날씨 수집 중...
  => 202312 평균 데이터 18건 저장 성공

[평균 작업 시작] 202401 ASOS 